# Mit `block-quantenschaltung` arbeiten

Dieses Notebook zeigt einen kleinen vollständigen Arbeitsablauf mit der Bibliothek:

1. einen Qiskit-Schaltkreis erstellen,
2. den resultierenden Statevector mit Aer und den beiden eigenen Simulatoren berechnen,
3. Messungen als Counts durchführen und
4. eine einzelne CNOT direkt auf einen Statevector anwenden.

Die Beispiele verwenden die Qiskit-Konvention, nach der Qubit `0` das niederwertigste Bit des Statevector-Index ist.

## Imports

Qiskit erstellt den Schaltkreis. `mock_simulate` verwendet Aer als Referenz, während `simulate` und `simulate_no_einsum` die eigenen Implementierungen aufrufen.

In [1]:
import numpy as np
import qiskit

import block_quantenschaltung as qs

## Einen Bell-Zustand erzeugen

Ein Hadamard-Gate auf Qubit `0` erzeugt eine Superposition. Die anschließende CNOT verschränkt Qubit `1` mit Qubit `0`. Der `save_statevector`-Marker wird für die Aer-Referenz benötigt; die eigenen Simulatoren entfernen ihn vor der Simulation automatisch.

In [2]:
circuit = qiskit.QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.save_statevector()

circuit.draw("text")

┌───┐      statevector 
q_0: ┤ H ├──■────────░──────
     └───┘┌─┴─┐      ░      
q_1: ─────┤ X ├──────░──────
          └───┘      ░

## Statevector mit drei Backends berechnen

Alle drei Frontends haben dieselbe `perform_sim()`-Schnittstelle. Das erste Backend ist Aer's Referenzimplementierung. Die anderen beiden unterscheiden sich in der internen Single-Qubit-Implementierung:

* `simulate` verwendet `numpy.einsum`.
* `simulate_no_einsum` verwendet die explizite, Numba-kompilierte Indexarithmetik.

In [3]:
aer_state = np.asarray(
    qs.mock_simulate(circuit, number_of_shots=1, return_statevector=True).perform_sim()
)
einsum_state = np.asarray(
    qs.simulate(circuit, number_of_shots=1, return_statevector=True).perform_sim()
)
no_einsum_state = np.asarray(
    qs.simulate_no_einsum(circuit, number_of_shots=1, return_statevector=True).perform_sim()
)

print("Aer:       ", aer_state)
print("einsum:    ", einsum_state)
print("no einsum: ", no_einsum_state)

Aer:        [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
einsum:     [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
no einsum:  [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


In [4]:
np.testing.assert_allclose(einsum_state, aer_state)
np.testing.assert_allclose(no_einsum_state, aer_state)
print("Beide eigenen Simulatoren stimmen mit Aer überein.")

Beide eigenen Simulatoren stimmen mit Aer überein.


## Messungen als Counts

Für Messungen wird `measure_all()` verwendet. Mit `return_statevector=False` liefern die Simulatoren ein Dictionary mit den beobachteten Bitstrings und ihrer Häufigkeit. Beim Bell-Zustand können nur `00` und `11` auftreten.

In [5]:
measurement_circuit = qiskit.QuantumCircuit(2)
measurement_circuit.h(0)
measurement_circuit.cx(0, 1)
measurement_circuit.measure_all()

shots = 1_000
counts = qs.simulate_no_einsum(
    measurement_circuit,
    number_of_shots=shots,
    return_statevector=False,
).perform_sim()

print(counts)
assert sum(counts.values()) == shots
assert set(counts) <= {"00", "11"}

{'00': 494, '11': 506}


## Eine CNOT direkt anwenden

Neben der Circuit-Schnittstelle können die Gate-Funktionen direkt mit einem Statevector verwendet werden. `apply_CNOT_reshape` nutzt die Speicherstruktur des Statevectors, um die betroffenen Amplitudenpaare zu vertauschen.

In [6]:
initial_state = np.zeros(4, dtype=complex)
initial_state[1] = 1.0  # |01> in der Qiskit-Little-Endian-Konvention

final_state = qs.apply_CNOT_reshape(
    control=0,
    target=1,
    state_vector=initial_state,
)

print(final_state)
np.testing.assert_allclose(final_state, [0, 0, 0, 1])


[0.+0.j 0.+0.j 0.+0.j 1.+0.j]


Damit ist der typische Ablauf abgedeckt: Qiskit erstellt den Circuit, ein Backend simuliert ihn, und die Ergebnisse können als Statevector oder als Mess-Counts weiterverarbeitet werden.